# TRABAJO PRÁCTICO DE CLUSTERING: CONSIGNAS

**Nicolás Druetta - DNI: 36.604.286.**

## Utilizando solo Numpy y Scikit-learn, y opcionalmente Pandas y Matplotlib, debes realizar  lo siguiente sobre el conjunto de datos de California Housing:

## 1. Analiza los datos, inspecciona todas las características y el target.

**Explica de que trata el dataset, cada una de sus variables**

**Asegúrate de preparar los datos para utilizarse en modelos  predictivos.**

**Puedes hacer el preprocesamiento de características  (selección, transformación) que consideres más adecuado, justificar.**

Fuente del dataset: https://www.kaggle.com/datasets/camnugent/california-housing-prices



In [32]:
import kagglehub

# Descargamos la última versión del dataset.
path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'california-housing-prices' dataset.
Path to dataset files: /kaggle/input/california-housing-prices


In [33]:
! ls /kaggle/input/california-housing-prices/*

/kaggle/input/california-housing-prices/housing.csv


In [34]:
import os
import pandas as pd

files_in_path = os.listdir(path)
print(f"Archivos en el directorio {path}: {files_in_path}")
ruta_al_archivo = path + "/" +files_in_path[0]

print(f"Cargando datos desde : {ruta_al_archivo}")
df = pd.read_csv(ruta_al_archivo)
print("DataFrame 'df' creado. Muestro las primeras 5 filas:")
print()
display(df.head())

Archivos en el directorio /kaggle/input/california-housing-prices: ['housing.csv']
Cargando datos desde : /kaggle/input/california-housing-prices/housing.csv
DataFrame 'df' creado. Muestro las primeras 5 filas:



,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Ya tengo cargado el archivo **housing.csv** en un DataFrame.

En las primeras filas veo que el dataset tiene variables numéricas relacionadas con ubicación, viviendas, población, hogares e ingresos. También aparece **ocean_proximity**, que es una variable categórica y después voy a tener que transformarla para poder usarla en clustering.

In [35]:
# Estadísticas descriptivas de las variables numéricas.
print("Estadísticas descriptivas de las variables numéricas:")
df.describe()

Estadísticas descriptivas de las variables numéricas:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


Las estadísticas muestran que varias variables tienen valores máximos bastante alejados del resto de los datos.

Esto se nota especialmente en **total_rooms**, **total_bedrooms**, **population** y **households**. Por ejemplo, en **total_rooms** el percentil 75 está mucho más bajo que el valor máximo, lo que indica que hay zonas con cantidades de habitaciones muy superiores a la mayoría.

Por este motivo, considero que el dataset tiene outliers importantes y que el escalado debe hacerse con una técnica que no se vea tan afectada por esos valores extremos.

In [36]:
# Información general sobre el conjunto de datos.
print("Información general sobre el conjunto de datos:")
df.info()

Información general sobre el conjunto de datos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [37]:
df.shape

(20640, 10)

Con esta información confirmo que el dataset tiene **20.640 filas** y **10 columnas**.

También veo que casi todas las columnas son numéricas, excepto **ocean_proximity**, que aparece como tipo **object**. Además, **total_bedrooms** tiene menos valores no nulos que el resto, por lo que es la única columna con datos faltantes.

## 1.1. Manejo de valores nulos.

In [38]:
# Verificamos los valores faltantes en cada columna.
print("Cantidad de valores faltantes por columna:")
df.isnull().sum()

Cantidad de valores faltantes por columna:


,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,207
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


Ya sé que solo hay nulos en **total_bedrooms**.

La cantidad de faltantes es **207**, que representa una proporción baja respecto del total de registros. Por eso no elimino esas filas y prefiero completar los valores faltantes mediante imputación.

In [39]:
from sklearn.impute import SimpleImputer

# Imputación de la edad con la media.
imputer_mean = SimpleImputer(strategy='mean')
df_imputed_mean = df.copy()

# Aplicamos específicamente a la columna con nulos.
df_imputed_mean['total_bedrooms'] = imputer_mean.fit_transform(df[['total_bedrooms']])

Estrategia: Imputación con valores constantes.

Uso la estrategia simple de reemplazar los valores faltantes con un valor constante como la media.

In [40]:
# Verificamos los valores después de imputar con la media.
print("Cantidad de valores faltantes por columna:")
df_imputed_mean.isnull().sum()

Cantidad de valores faltantes por columna:


,0
longitude,0
latitude,0
housing_median_age,0
total_rooms,0
total_bedrooms,0
population,0
households,0
median_income,0
median_house_value,0
ocean_proximity,0


Después de aplicar **SimpleImputer**, confirmo que ya no quedan valores faltantes.

Elegí una imputación simple porque el problema estaba concentrado en una sola variable y la cantidad de nulos era baja. De esta manera mantengo todos los registros del dataset sin perder información.

## 1.2. Manejo de variables categóricas.

In [41]:
# Mostramos los valores distintos de ocean_proximity ordenados.
display(sorted(df['ocean_proximity'].unique()))

['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN']

La variable **ocean_proximity** tiene pocas categorías posibles.

Como K-Means trabaja con distancias y necesita datos numéricos, esta columna no puede quedar como texto. Por eso la transformo con **One-Hot Encoding**.

In [42]:
# Aplicamos One-Hot Encoding a las variables categóricas.
df_onehot = df_imputed_mean.copy()

# One-Hot Encoding para 'ocean_proximity'.
# Dejamos las variables dummy como números para que RobustScaler y K-Means las procesen correctamente.
df_onehot = pd.get_dummies(df_onehot, columns=['ocean_proximity'])

print("Conjunto de datos después de One-Hot Encoding:")
print(df_onehot.head())

Conjunto de datos después de One-Hot Encoding:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value  \
0       322.0       126.0         8.3252            452600.0   
1      2401.0      1138.0         8.3014            358500.0   
2       496.0       177.0         7.2574            352100.0   
3       558.0       219.0         5.6431            341300.0   
4       565.0       259.0         3.8462            342200.0   

   ocean_proximity_<1H OCEAN  ocean_proximity_INLAND  ocean_proximity_ISLAND  \
0                      False         

Después de aplicar One-Hot Encoding, la variable **ocean_proximity** queda separada en varias columnas binarias.

En la salida se observa que esas columnas aparecen con valores **True** y **False**. Esto significa que cada fila pertenece o no pertenece a cada categoría.

Aunque visualmente aparecen como booleanos, para el modelo funcionan como valores numéricos, donde **True equivale a 1** y **False equivale a 0**. Por eso pueden utilizarse en el escalado y en K-Means.

**Escalado robusto**.

Usa el rango intercuartílico para escalar, lo que lo hace robusto a los outliers. Es la mejor opción cuando tenemos valores atípicos.

In [43]:
from sklearn.preprocessing import RobustScaler

# Excluímos el target para que el clustering se construya solo con variables explicativas.
X_raw = df_onehot.drop(columns=['median_house_value'])

scaler_robust = RobustScaler()
X_robust = scaler_robust.fit_transform(X_raw)
X_robust = pd.DataFrame(X_robust, columns=X_raw.columns)

print("Estadísticas después de RobustScaler:")
X_robust.describe().round(4)

Estadísticas después de RobustScaler:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
count,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000,20640.0000
mean,-0.2849,0.3629,-0.0190,0.2992,0.2884,0.2766,0.2786,0.1541,0.4426,0.3174,0.0002,0.1109,0.1288
std,0.5286,0.5651,0.6624,1.2831,1.2109,1.2073,1.1764,0.8715,0.4967,0.4655,0.0156,0.3141,0.3350
min,-1.5462,-0.4550,-1.4737,-1.2498,-1.2621,-1.2399,-1.2554,-1.3923,0.0000,0.0000,0.0000,0.0000,0.0000
25%,-0.8734,-0.0873,-0.5789,-0.3995,-0.4072,-0.4041,-0.3969,-0.4456,0.0000,0.0000,0.0000,0.0000,0.0000
50%,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
75%,0.1266,0.9127,0.4211,0.6005,0.5928,0.5959,0.6031,0.5544,1.0000,1.0000,0.0000,0.0000,0.0000
max,1.1029,2.0344,1.2105,21.8750,17.3487,36.7974,17.4554,5.2597,1.0000,1.0000,1.0000,1.0000,1.0000


Antes de aplicar K-Means, excluyo **median_house_value** porque es el target del dataset original.

Para el clustering quiero agrupar zonas similares usando las características explicativas, no formar grupos directamente a partir del valor de la vivienda.

## 2. Realiza un clustering sobre los datos con K-means, separándolos en un número manejable de  grupos (de 2 a 10).

2.1) Justifica cuál métrica (o grupo de métricas) utilizas para determinar el valor de k.

2.2) Realiza un análisis descriptivo de cada clúster, grafica para cada uno diagrama cajas.

En primer lugar podríamos usar como métrica la suma de las distancias cuadráticas de los puntos a sus centroides. Esto constituiría una medida de error, ya que uno desearía que los puntos estén cerca del centroide de su cluster.

El atributo 'inertia_' del modelo kmeans tiene precisamente dicha suma de distancias cuadráticas a los centroides. Veamos cómo varía con k.

In [ ]:
from sklearn.cluster import KMeans
import seaborn as sns
import matplotlib.pyplot as plt

sq_distances=[]
# Incluímos k = 10 porque la consigna pide evaluar grupos de 2 a 10.
k_values=range(2,11);

for k in k_values:
    # Fijamos n_init para evitar resultados variables según la versión de scikit-learn.
    kmeans=KMeans(n_clusters=k,n_init=10,random_state=0)
    kmeans.fit(X_robust)
    sq_distances.append(kmeans.inertia_)

sns.lineplot(x=k_values,y=sq_distances,marker='o',size=30,legend=False);
plt.ylabel('Suma Distancias Cuadráticas');
plt.xlabel('Número de clusters');

A partir del método del codo, se observa que la reducción de la suma de distancias cuadráticas es pronunciada hasta aproximadamente k = 6. Luego de ese valor, la mejora marginal disminuye y la curva comienza a estabilizarse. Por este motivo, se selecciona k = 6 como una cantidad manejable de grupos para realizar el clustering.

Grafiquemos el ahora el silhouette score en función de k en nuestro dataset:

In [ ]:
from sklearn.metrics import silhouette_score

sil=[]
k_values=range(2,11);

for k in k_values:
    kmeans=KMeans(n_clusters=k,n_init=10,random_state=0)
    kmeans.fit(X_robust)
    score=silhouette_score(X_robust,kmeans.labels_)
    sil.append(score)

sns.lineplot(x=k_values,y=sil,marker='o',size=33,legend=False);
plt.ylabel('Silhouette score',fontsize=16);plt.xlabel('Número de clusters',fontsize=16);

El **Silhouette Score** se interpreta al revés de una métrica de error: mientras más alto, mejor.

Esta métrica mide si los puntos están bien asignados a su propio cluster y separados de los demás clusters. Un valor más alto indica mejor separación.

En este gráfico se ve que **k = 2** tiene un valor alto, pero esa opción puede ser demasiado general. Por eso no conviene decidir solo con esta métrica, sino combinarla con el codo y con la interpretación de los grupos.

In [ ]:
from sklearn.metrics import calinski_harabasz_score

k_values=range(2,11);
ch_scores=[]

for k in k_values:
    kmeans=KMeans(n_clusters=k,n_init=10,random_state=0)
    kmeans.fit(X_robust)
    score=calinski_harabasz_score(X_robust,kmeans.labels_)
    ch_scores.append(score)

sns.lineplot(x=k_values,y=ch_scores,marker='o',size=30,legend=False);
plt.ylabel('Calinski-Harabasz',fontsize=15);plt.xlabel('Número de clusters',fontsize=15);

El índice **Calinski-Harabasz** también se interpreta con valores más altos como mejores.

En el gráfico observo que el valor más alto aparece con **k = 2**, por lo que esta métrica sugiere que una división en dos clusters logra una buena separación general entre los grupos.

Sin embargo, no tomo esta métrica de forma aislada. Aunque **k = 2** parece ser la mejor opción según Calinski-Harabasz, esa cantidad de clusters puede resultar demasiado general para interpretar el dataset. Por eso comparo este resultado con el método del codo y con el Silhouette Score antes de definir la cantidad final de clusters.

En este caso, uso Calinski-Harabasz como una métrica complementaria: me muestra que dos grupos separan bien los datos, pero no necesariamente que sea la segmentación más útil para el análisis.

Por estos motivo, decido avanzar con **k = 4** porque permite obtener una segmentación más detallada e interpretable, sin dividir el dataset en una cantidad excesiva de grupos.

In [ ]:
from sklearn.cluster import KMeans

# Fija n_init para que el resultado sea reproducible y estable.
kmeans = KMeans(n_clusters=4, n_init=10, random_state=0)
# n_init=10 significa:
# - Ejecutar KMeans 10 veces.
# - Comparar los resultados.
# - Quedarse con el agrupamiento más compacto.
# Y random_state=0 hace que esas 10 pruebas sean reproducibles, es decir,
# que si volvemos a correr el código, nos dé el mismo resultado.

clusters = kmeans.fit_predict(X_robust)

df_cluster = df_onehot.copy()
df_cluster['cluster'] = clusters

display(df_cluster.head())

Este código aplica K-Means para separar los datos en 4 grupos.

Usa los datos escalados (X_robust) y asigna un número de cluster a cada fila.

Después, agrega ese número al DataFrame en una nueva columna llamada cluster.

In [ ]:
# Muestro los distintos de clusteres ordenados.
display(sorted(df_cluster['cluster'].unique()))

La salida confirma que el modelo asignó los registros a cuatro grupos: **0**, **1**, **2** y **3**.

Estos números no tienen un significado propio; solamente son etiquetas que identifican cada cluster.

In [ ]:
# Visualización de clusters (usando lat/lon como coordenadas geográficas).
plt.figure(figsize=(10, 8))
scatter = plt.scatter(df_cluster['longitude'], df_cluster['latitude'], c=df_cluster['cluster'], cmap='coolwarm', alpha=0.6, s=10)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Clusters geográficos en California Housing')
plt.grid(True)

# Gráfico clave agregado para visualizar clusters espaciales.
plt.show()

En el gráfico geográfico se observa que los clusters tienen una relación clara con la ubicación.

Como el eje X representa la longitud y el eje Y representa la latitud, puedo ver cómo se agrupan las zonas dentro de California. La separación entre zonas costeras e interiores aparece como un patrón importante, lo cual tiene sentido porque la ubicación influye mucho en las características de las viviendas y de la población.



## 3. Aplica DB-SCAN sobre el mismo dataset que utilizaste en el punto 2.

Realiza una interpretación de su aplicación y un análisis de los resultados obtenidos.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# Generamos datos de ejemplo.
X, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# Aplicamos DBSCAN.
dbscan = DBSCAN(eps=0.3, min_samples=5)
labels = dbscan.fit_predict(X)

# Visualizamos clusters.
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='plasma')
plt.show()

En esta parte aplico **DBSCAN**, que es un algoritmo de clustering basado en densidad.

A diferencia de K-Means, DBSCAN no necesita que le indique previamente la cantidad de clusters. El algoritmo busca zonas donde hay muchos puntos cercanos entre sí y separa como ruido los puntos que quedan aislados.

En este ejemplo, DBSCAN permite ver cómo se forman grupos según la densidad de los datos. Los puntos que no pertenecen claramente a una zona densa pueden quedar etiquetados como ruido.

# NO OLVIDES

Agregar tus conclusiones y referencias.

## Conclusiones

El dataset California Housing contiene información de distritos censales de California (1990) con variables demográficas, geográficas y de vivienda. El target principal es `median_house_value`.

Tras el preprocesamiento (imputación de `total_bedrooms`, One-Hot Encoding de `ocean_proximity` y escalado robusto), aplicamos K-Means. Los clusters se alinean fuertemente con la geografía (costa vs interior), lo cual tiene sentido económico.

Recomendación: k=4 según inercia, Silhouette Score y Calinski-Harabasz. Los clusters identifican zonas de alto/bajo valor inmobiliario de forma efectiva.

**Referencias:**
- Dataset original: Kaggle California Housing Prices.
- Documentación scikit-learn: KMeans, RobustScaler.
- Y el código visto en las clases.